# Personne C — Aéroports (J3 §4 + `OPTIMIZE`, J4 segmentation K-Means, onglet « Aéroports »)

**Question** : les aéroports ont-ils des profils de retard différents ? (suite directe de l'ACP de l'étape 5 de J2)

| Livrable | Où |
|---|---|
| J3 §4 : table Gold `airport_profile` (`ROW_NUMBER() OVER (PARTITION BY ...)`) | `/lakehouse/gold/airport_profile` |
| J3 §4 : `OPTIMIZE` de la table Silver + explication | ce notebook |
| J4 : segmentation K-Means des aéroports | `/lakehouse/gold/airport_segments` |
| Onglet « Aéroports » du dashboard | `app/sections/aeroports.py` |

**Règles communes** (détail dans `docs/repartition_J3_J4.md`)
- Ne modifiez que ce notebook et votre fichier `app/sections/aeroports.py` : aucun conflit git possible.
- Les tables Gold respectent **exactement** les noms et colonnes du contrat : le dashboard et l'assemblage final en dépendent. `save_gold(...)` le vérifie.
- Taux de retard = part des vols **arrivés** avec `arrival_delay >= 15` (vue `v_arrives`). Taux d'annulation = sur **tous** les vols (`v_silver`).
- Octobre a des codes aéroport non valides : pour toute analyse par aéroport, filtrez `airport_code_valid`.
- Chaque résultat est suivi d'une interprétation en Markdown (comme en J2). Committez le notebook **exécuté**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("C_aeroports").getOrCreate()
sns.set_theme(style="whitegrid", context="notebook")

SILVER_MAIN_PATH = "/lakehouse/silver/flights_enriched"
spark.read.format("delta").load(SILVER_MAIN_PATH).createOrReplaceTempView("v_silver")
# Vols arrivés : les seuls qui ont un arrival_delay (même périmètre que l'EDA de J2)
spark.sql("""
    SELECT * FROM v_silver
    WHERE NOT is_cancelled AND NOT is_diverted AND arrival_delay IS NOT NULL
""").createOrReplaceTempView("v_arrives")
print("v_silver :", spark.table("v_silver").count(), "vols | v_arrives :", spark.table("v_arrives").count(), "vols arrivés")


def save_gold(df, name, contrat, n_files=1):
    """Vérifie les colonnes du contrat (docs/repartition_J3_J4.md) puis écrit /lakehouse/gold/<name> en Delta."""
    manquantes = [c for c in contrat if c not in df.columns]
    assert not manquantes, f"Colonnes du contrat manquantes pour {name} : {manquantes}"
    path = f"/lakehouse/gold/{name}"
    autres = [c for c in df.columns if c not in contrat]
    (df.select(*contrat, *autres).coalesce(n_files)
       .write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(path))
    n = spark.read.format("delta").load(path).count()
    print(f"Table Gold écrite : {path} ({n:,} lignes)")

## J3 §4 — Table Gold `airport_profile`

Une ligne par aéroport de départ (`airport_code_valid`, au moins 300 vols) :

| colonne | définition |
|---|---|
| `origin_airport`, `origin_city`, `origin_state`, `origin_lat`, `origin_lon` | identité et position |
| `n_vols` | vols au départ (`v_silver`) |
| `taux_retard` | % de vols arrivés avec `arrival_delay >= 15` |
| `taux_annulation` | % de vols annulés |
| `roulage_median` | médiane de `taxi_out` |
| `rang_etat` | `ROW_NUMBER() OVER (PARTITION BY origin_state ORDER BY taux_retard)` (1 = le moins en retard de l'État) |
| `meilleure_compagnie`, `taux_retard_meilleure_compagnie` | compagnie au plus faible taux de retard au départ de cet aéroport, parmi celles avec ≥ 50 vols : `ROW_NUMBER() OVER (PARTITION BY origin_airport ORDER BY taux_retard) = 1` |

In [ ]:
AIRPORT_PROFILE_COLS = ["origin_airport", "origin_city", "origin_state", "origin_lat", "origin_lon", "n_vols",
                        "taux_retard", "taux_annulation", "roulage_median", "rang_etat",
                        "meilleure_compagnie", "taux_retard_meilleure_compagnie"]

df_gold_3 = spark.sql("""
    WITH aeroports AS (
        SELECT ...   -- TODO : indicateurs par aéroport (HAVING COUNT(*) >= 300)
    ),
    compagnie_par_aeroport AS (
        SELECT ...   -- TODO : taux de retard par (aéroport, compagnie), >= 50 vols, ROW_NUMBER
    )
    SELECT ...       -- TODO : jointure + rang_etat
""")
save_gold(df_gold_3, "airport_profile", AIRPORT_PROFILE_COLS)
df_gold_3.orderBy("taux_retard").show(10, truncate=False)

### `OPTIMIZE` de la table Silver

Lancez `OPTIMIZE` sur Silver, comparez le nombre de fichiers avant / après (`DESCRIBE DETAIL` → `numFiles`) et expliquez en Markdown pourquoi beaucoup de petits fichiers pénalisent le NameNode HDFS (une entrée en mémoire par fichier et par bloc, blocs de 16 Mo ici).

In [ ]:
avant = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").select("numFiles", "sizeInBytes").first()
spark.sql(f"OPTIMIZE delta.`{SILVER_MAIN_PATH}`").show(truncate=False)
apres = spark.sql(f"DESCRIBE DETAIL delta.`{SILVER_MAIN_PATH}`").select("numFiles", "sizeInBytes").first()
print(f"Fichiers : {avant.numFiles} -> {apres.numFiles}")

**Explication** *(à compléter)*.

## J4 — Segmentation K-Means des aéroports → `airport_segments`

Reprenez **les 9 variables de l'ACP** (J2, étape 5 : copiez son agrégation) : `taux_retard`, `taux_annulation`, `roulage_median`, `log_trafic`, `distance_mediane`, `part_depart_soir`, `part_systeme`, `part_meteo`, `part_avion_retarde`.
1. `np.log1p` sur les variables positives asymétriques (à vérifier avec la skewness), puis `StandardScaler`.
2. Choix de k : méthode du coude (inertie) **et** score de silhouette pour k = 2 à 8.
3. Médianes par cluster → un **nom métier** par segment (ex. « hub congestionné », « régional perturbé », « réseau à rotations »…).
4. Vérifiez le lien avec l'ACP : projetez les clusters sur CP1 × CP2.

Contrat : `origin_airport`, `origin_city`, `origin_lat`, `origin_lon`, `n_vols`, les 9 variables, `cluster_id`, `segment`.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

FEATURES = ["taux_retard", "taux_annulation", "roulage_median", "log_trafic", "distance_mediane",
            "part_depart_soir", "part_systeme", "part_meteo", "part_avion_retarde"]
AIRPORT_SEGMENTS_COLS = ["origin_airport", "origin_city", "origin_lat", "origin_lon", "n_vols", *FEATURES, "cluster_id", "segment"]

# TODO 1 : agrégation Spark par aéroport (reprise de l'ACP J2), .toPandas()
entities_pd = ...

# TODO 2 : log1p + StandardScaler, coude + silhouette, KMeans(n_clusters=k, n_init=20, random_state=42)

# TODO 3 : médianes par cluster, noms métier dans entities_pd["segment"]

save_gold(spark.createDataFrame(entities_pd), "airport_segments", AIRPORT_SEGMENTS_COLS)

**Interprétation des segments** *(à compléter)* : une phrase par segment, avec des exemples d'aéroports.